# 03 - Fine-Tuning de ResNet50 para Classificacao de Especies com Ruido Artificial (5%)

Reproduz o pipeline de adaptacao a ruido de fragmentos curtos (512 bp) que sustenta os checkpoints e resultados das **Tabelas 1, 3 e 4** do manuscrito.
- Treinamento com corrupcao artificial de sequencias (epsilon = 0.05) aplicada em treino e validacao (AD-009, Notas 21-23).
- Carrega o checkpoint de especie base (`resnet_clean_species`) com 13.683 classes (AD-001).
- Suporta dois modos:
  1. `MODE = "historical_reproduction"`: hiperparametros historicos (`dropout=0.5`, `lr=5e-5`, `AdamW`, `CosineAnnealingLR`, `label_smoothing=0.1`, selecao da epoca 14 final `finetune_noise_14.pt`).
  2. `MODE = "controlled_parity"`: protocolo pareado de `configs/controlled/baseline_pair.yaml` (AD-016) com telemetria de recursos (`ResourceProfiler`).
- Desacoplamento via `src/viral_cnn/` sem `import *` e salvamento seguro em `results_reproduction/checkpoints/` sem alterar os pesos historicos (AGENTS.md).


In [1]:
import os
import sys
from pathlib import Path

# Resolucao autonoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import pandas as pd
import numpy as np

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
)
from viral_cnn.models.registry import (
    REGISTERED_VARIANTS,
    build_registered_model,
)
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.training.config import ControlledExperimentConfig
from viral_cnn.training.trainer import ControlledTrainer

# 1. Initialize Runtime and Environment (AD-017)
ctx = setup_notebook_runtime()
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")

MODE = "historical_reproduction"  # Options: "historical_reproduction", "controlled_parity"
NOISE_RATE = 0.05
EPOCHS = 1 if ctx.is_smoke_test else 15
MAX_BATCHES = 2 if ctx.is_smoke_test else None


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace


In [2]:
# 2. Setup DataLoaders for Train and Validation Partitions
if ctx.is_smoke_test:
    ensure_smoke_test_artifacts(ctx.data_root)

output_dir = Path("results_reproduction/n03_finetune/resnet50")
checkpoint_dir = Path("results_reproduction/checkpoints/resnet50_noise")
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])

train_files = resolve_split_files(species_parquet_dir, "train")
val_files = resolve_split_files(species_parquet_dir, "validation")

train_ds = ParquetBatchIterableDataset(
    train_files,
    batch_size=32 if ctx.is_smoke_test else 4096,
    shuffle=True,
    seed=ctx.seed,
    split="train",
)
train_loader = create_optimized_dataloader(
    train_ds,
    is_smoke_test=ctx.is_smoke_test,
    device=ctx.device,
)

val_ds = ParquetBatchIterableDataset(
    val_files,
    batch_size=32 if ctx.is_smoke_test else 4096,
    shuffle=False,
    seed=ctx.seed,
    split="validation",
)
val_loader = create_optimized_dataloader(
    val_ds,
    is_smoke_test=ctx.is_smoke_test,
    device=ctx.device,
)

print(f"DataLoaders ready. Train files: {len(train_files)}, Val files: {len(val_files)}")


[LOG_PATH] Loading DataLoader/ParquetDataset from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet/train/part.0.parquet ... (256 files total) | 256 partitions, 13650160 samples, batch_size=4096


DataLoaders ready. Train files: 256, Val files: 256


[LOG_PATH] Loading DataLoader/ParquetDataset from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet/validation/part.0.parquet ... (256 files total) | 256 partitions, 1706271 samples, batch_size=4096


In [3]:
# 3. Model and Controlled Configuration
base_model_name = "resnet_clean_species"
print(f"Loading base model: {base_model_name}")
model = build_registered_model(
    base_model_name,
    data_root=ctx.data_root,
    device=ctx.device,
)

if hasattr(torch, "compile") and ctx.device.type == "cuda":
    try:
        model = torch.compile(model, dynamic=True)
        print("Model compiled with torch.compile(dynamic=True)")
    except Exception as e:
        print(f"torch.compile skipped: {e}")

config = ControlledExperimentConfig(
    architectures=("resnet50_1d",),
    num_classes=13683,
    learning_rate=5e-5,
    weight_decay=1e-4,
    warmup_steps=0 if MODE == "historical_reproduction" else 20,
    label_smoothing=0.1,
    dropout_rate=0.5 if MODE == "historical_reproduction" else 0.5,
    noise_rate=NOISE_RATE,
    checkpoint_metric="last_step" if MODE == "historical_reproduction" and "resnet" in base_model_name else "best_val_macro_f1",
    mode=MODE,
)

trainer = ControlledTrainer(
    model=model,
    config=config,
    device=ctx.device,
)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    trainer.optimizer,
    T_max=max(1, EPOCHS * len(train_ds)),
    eta_min=1e-7,
)
print("Trainer and CosineAnnealingLR configured.")

# Pre-training validation evaluation (Epoch 0)
print("--- Pre-training Evaluation (Epoch 0) ---")
prev_noise = config.noise_rate
trainer.config.noise_rate = 0.0
val_clean_pre = trainer.evaluate_epoch(val_loader, max_batches=2 if ctx.is_smoke_test else 10)
trainer.config.noise_rate = NOISE_RATE
val_noise_pre = trainer.evaluate_epoch(val_loader, max_batches=2 if ctx.is_smoke_test else 10)
trainer.config.noise_rate = prev_noise
print(f"Epoch 0 Pre-Train Val Clean (eps=0.0): Acc={val_clean_pre['val_acc']:.4f} | Macro-F1={val_clean_pre['val_f1_macro']:.4f}")
print(f"Epoch 0 Pre-Train Val Noise (eps={NOISE_RATE}): Acc={val_noise_pre['val_acc']:.4f} | Macro-F1={val_noise_pre['val_f1_macro']:.4f}")


Loading base model: resnet_clean_species


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt


Model compiled with torch.compile(dynamic=True)
Trainer and CosineAnnealingLR configured.


In [4]:
# 4. Noise-Aware Fine-Tuning Execution
import shutil
import time

print(f"--- Starting resnet50_1d Fine-Tuning (Noise Rate: {NOISE_RATE*100}%, Epochs: {EPOCHS}, Mode: {MODE}) ---")

last_ckpt = checkpoint_dir / "finetune_noise_last.pt"
start_epoch = 1
training_history = []
best_val_f1 = 0.0

if last_ckpt.exists() and not ctx.is_smoke_test:
    print(f"Found existing checkpoint at {last_ckpt}. Resuming training...")
    start_epoch, training_history = trainer.load_resume_checkpoint(last_ckpt, scheduler=lr_scheduler)
    if training_history:
        best_val_f1 = max(r.get("val_f1_macro", 0.0) for r in training_history)
    print(f"Resumed from Epoch {start_epoch} with {len(training_history)} history records. Best Val Macro-F1: {best_val_f1:.4f}")

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.perf_counter()
    train_res = trainer.train_epoch(
        train_loader=train_loader,
        max_batches=MAX_BATCHES,
        scheduler=lr_scheduler,
    )
    val_res = trainer.evaluate_epoch(
        val_loader=val_loader,
        max_batches=MAX_BATCHES,
    )
    dt = time.perf_counter() - t0
    
    epoch_record = {
        "epoch": epoch,
        "train_loss": train_res["train_loss"],
        "train_acc": train_res["train_acc"],
        "train_f1": train_res["train_f1"],
        "train_f1_weighted": train_res["train_f1_weighted"],
        "val_loss": val_res["val_loss"],
        "val_acc": val_res["val_acc"],
        "val_f1_macro": val_res["val_f1_macro"],
        "val_f1_weighted": val_res["val_f1_weighted"],
        "learning_rate": train_res["learning_rate"],
        "epoch_seconds": round(dt, 2),
    }
    training_history.append(epoch_record)
    
    print(
        f"Epoch {epoch}/{EPOCHS} ({dt:.1f}s) -> "
        f"Train Loss: {train_res['train_loss']:.4f} | Train Acc: {train_res['train_acc']:.4f} | "
        f"Val Loss: {val_res['val_loss']:.4f} | Val Acc: {val_res['val_acc']:.4f} | "
        f"Val Macro-F1: {val_res['val_f1_macro']:.4f} | LR: {train_res['learning_rate']:.2e}"
    )
    
    # Serialize checkpoint once per epoch and copy to numbered/best targets
    last_ckpt = checkpoint_dir / "finetune_noise_last.pt"
    trainer.save_epoch_checkpoint(
        epoch=epoch,
        metrics=epoch_record,
        save_path=last_ckpt,
        scheduler=lr_scheduler,
        training_history=training_history,
    )
    shutil.copyfile(last_ckpt, checkpoint_dir / f"finetune_noise_{epoch}.pt")
    
    if val_res["val_f1_macro"] > best_val_f1:
        best_val_f1 = val_res["val_f1_macro"]
        shutil.copyfile(last_ckpt, checkpoint_dir / "finetune_noise_best.pt")

print("Fine-tuning completed successfully.")


--- Starting resnet50_1d Fine-Tuning (Noise Rate: 5.0%, Epochs: 15, Mode: historical_reproduction) ---
Epoch 1/15 (1306.3s) -> Train Loss: 9.8514 | Train Acc: 0.0010 | Val Loss: 8.7828 | Val Acc: 0.0024 | Val Macro-F1: 0.0011
Epoch 2/15 (906.7s) -> Train Loss: 8.5071 | Train Acc: 0.0039 | Val Loss: 8.4468 | Val Acc: 0.0081 | Val Macro-F1: 0.0070
Epoch 3/15 (907.5s) -> Train Loss: 8.2217 | Train Acc: 0.0065 | Val Loss: 8.3127 | Val Acc: 0.0141 | Val Macro-F1: 0.0118
Epoch 4/15 (908.1s) -> Train Loss: 8.0378 | Train Acc: 0.0097 | Val Loss: 8.2135 | Val Acc: 0.0211 | Val Macro-F1: 0.0185
Epoch 5/15 (906.3s) -> Train Loss: 7.8529 | Train Acc: 0.0143 | Val Loss: 8.1323 | Val Acc: 0.0327 | Val Macro-F1: 0.0316
Epoch 6/15 (908.6s) -> Train Loss: 7.6990 | Train Acc: 0.0192 | Val Loss: 8.0700 | Val Acc: 0.0398 | Val Macro-F1: 0.0394
Epoch 7/15 (908.4s) -> Train Loss: 7.5808 | Train Acc: 0.0237 | Val Loss: 8.0351 | Val Acc: 0.0456 | Val Macro-F1: 0.0454
Epoch 8/15 (906.8s) -> Train Loss: 7.4888 

In [5]:
# 5. Export Metrics History and Run Manifest
history_df = pd.DataFrame(training_history)
print("=== Training History ===")
print(history_df.to_string())
history_csv = output_dir / "training_history.csv"
history_json = output_dir / "training_history.json"
history_df.to_csv(history_csv, index=False)
history_df.to_json(history_json, orient="records", indent=2)

final_epoch_metrics = training_history[-1] if training_history else {}
manifest = write_notebook_manifest(
    notebook_id="resnet50_finetune_species",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    metrics_summary={
        "mode": MODE,
        "epochs": EPOCHS,
        "noise_rate": NOISE_RATE,
        "final_epoch_metrics": final_epoch_metrics,
        "best_val_macro_f1": best_val_f1,
    },
    checkpoints={
        "base_model": base_model_name,
        "saved_checkpoint_last": str(checkpoint_dir / "finetune_noise_last.pt"),
    },
    input_files=train_files + val_files,
    seed=ctx.seed,
    output_json_path=output_dir / "manifest.json",
    output_yaml_path=output_dir / "manifest.yaml",
)
print("Artifacts and manifest saved to", output_dir)


Artifacts and manifest saved to results_reproduction/n03_finetune/resnet50
